# Pre-processing

Module imports

In [1]:
import multiprocessing as mp
import os
import sys
from functools import partial

import numpy as np
import trimesh
from numba import njit
from scipy.spatial import KDTree
from tqdm import tqdm

Adding PATH 

In [4]:
# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root added to sys.path


Parsing `.gro` file (slightly slower method by accessing data from pandas df)

In [5]:
# Import module
from utils import gro_processing as gp

# Data directory can be accessed due to root PATH we set previously
file = os.path.join(project_root, 'data/npt-HK4.gro')

# Extracts data from .gro file into DataFrame (unsorted)
data, title, num_atoms, box = gp.read_gro(file) 

# Parameter
box_length = box[0]

# Create multi-index DataFrame (sorted)
df_gro = gp.dataframe_gro(data, box_length, positions=True, velocities=False, oxygen_midpoints=False)

# Dictionary of molecules {res_id: [(atom_name, np.array([x, y, z])), ...]}
molecules = {}

for res_id in df_gro.index.get_level_values('res_id').unique():
    residue_data = df_gro.xs(res_id, level='res_id')
    
    # Optional: verify we have 84 atoms
    if len(residue_data) != 84:
        print(f"Warning: res_id {res_id} has {len(residue_data)} atoms, expected 84")
    
    # Using itertuples() - much faster for large DataFrames
    atom_list = [
        (row.Index, np.array([row.x, row.y, row.z]) * 10)
        for row in residue_data.itertuples()
    ]
    
    molecules[res_id] = atom_list
    
box = box * 10
# molecules # display 

# Setup

Various configs and atom radii data for constructing molecule mesh

In [4]:
# --- CONFIG ---
gro_path = "./1mol.gro"                   
sphere_radius_scale = 2.0                # balls (atoms): 2.0 × van der Waals radius   # idk why 1.5 :(
bond_radius = 0.1                        # sticks (bonds): cylinder radius in Å
sphere_subdiv = 2                        # atom sphere detail (2 is moderate)

# --- Radii (Å) ---
vdw = {"H":1.20,"C":1.70,"N":1.55,"O":1.52,"F":1.47,"P":1.80,"S":1.80,"Cl":1.75,"Na":2.27,"K":2.75,"Ca":2.31}
cov = {"H":0.31,"C":0.76,"N":0.71,"O":0.66,"F":0.57,"P":1.07,"S":1.05,"Cl":1.02,"Na":1.66,"K":2.03,"Ca":1.74}

# --- Element inference ---
hash_atom = {"OW": "O", "HW": "H", "HW1": "H", "HW2": "H"} # Atom name aliases
color_map = {'O': np.array([220, 20, 60, 255], dtype=np.uint8),     # Crimson Red
             'N': np.array([65, 105, 225, 255], dtype=np.uint8),    # Royal Blue
             'C': np.array([128, 128, 128, 255], dtype=np.uint8),   # Gray
             'H': np.array([230, 230, 230, 255], dtype=np.uint8),   # Light Gray
             }

def infer_element(atomname):
    # common water aliases
    if atomname in hash_atom: 
        return hash_atom[atomname]
    
    # simple: first letter, capitalize second if lowercase
    a = ''.join([c for c in atomname if c.isalpha()])   
    
    # join() joins items in an iterable into one string, '' is specified as the separator.
    # isalpha() method returns True if all the characters are alphabet letters (a-z).

    if a == '': 
        return "C"

    if len(a) >= 2 and a[1].islower(): 
        return (a[0]+a[1]).capitalize()
    
    return a[0].upper()

Helper function for MIC related calculations

In [6]:
# --- Minimum Image Convention (vector) ---
@njit(cache=True, fastmath=True)
def mic_vector(dx, box_length):
    """Return minimum-image displacement for vector dx under PBC."""
    return dx - np.rint(dx / box_length) * box_length

@njit(cache=True, fastmath=True)
def mic_distance(a, b, box_length):
    """Return MIC distance between two 3D points a,b."""
    return np.linalg.norm(mic_vector(b - a, box_length))


x = np.array([8.0, 9.0, 10.0])
y = np.array([10.0, 10.0, 10.0])
_ = mic_vector(x, y) 
_ = mic_distance(x, np.array([1.0, 2.0, 3.0]), y)

# Generating molecule meshes

This part has 3 function with the following uses:
- Drawing a single molecule mesh
- Looping through all 1501 molecules to draw mesh
- Saving the meshes into `.ply` file as binary

## Ball-and-stick molecule model

Function to build a single molecule mesh

In [8]:
# --- Build ball-and-stick with PBC ---
def build_molecule_ballstick(coords, elements, vdw, cov, box_length, sphere_radius_scale=0.3, sphere_subdiv=2, bond_radius=0.1):
    """
    Build trimesh ball-and-stick model under periodic boundary conditions.
    """

    # radii arrays
    vdw_r = np.array([vdw.get(e, 1.70) for e in elements])
    cov_r = np.array([cov.get(e, 0.76) for e in elements])
    
    # number of atoms
    n = len(coords)

    
    # --- Process spheres using chunking--- 
    # Base icosphere (unit radius)
    base_sphere = trimesh.creation.icosphere(subdivisions=sphere_subdiv, radius=1.0)
    # Precompute position, radii and number of vertices for each sphere
    coords_wrapped = np.mod(coords, box_length)
    scaled_radii = vdw_r * sphere_radius_scale  
    num_sphere_vertices = len(base_sphere.vertices)
    # Chunk (pre-allocate) array to hold icosphere object
    meshes_sphere = np.empty(n, dtype=object)
    
    for idx, (pos, r) in enumerate(zip(coords_wrapped, scaled_radii)):
        sphere = base_sphere.copy()
        sphere.apply_scale(r)
        sphere.apply_translation(pos)
        sphere_color = color_map.get(elements[idx], color_map.get('C')) # element not found default to C (grey)
        sphere.visual.vertex_colors = np.tile(sphere_color, (num_sphere_vertices, 1))
        meshes_sphere[idx] = sphere 
        
        
    
    # --- Process cylinder with numba --- 
    # For small molecules roughly < 1500 atoms, brute force is most efficient
    # Base cylinder (for number of faces and color assignment)
    base_cyl = trimesh.creation.cylinder(radius=1.0, height=1.0, sections=24)
    num_bond_faces = len(base_cyl.faces)
    bond_color = color_map.get('H') # light grey
    
    meshes_cylinder = []
    if n < 1500: # brute force
        for i in range(n):
            for j in range(i+1, n):
                d = mic_distance(coords[i], coords[j], box_length)
                thr = 1.2 * (cov_r[i] + cov_r[j])
                if d < thr:
                    # Unwrap j relative to i
                    disp = mic_vector(coords[j] - coords[i], box_length)
                    pos_i = np.mod(coords[i], box_length)
                    pos_j = pos_i + disp  # may fall outside box but correct bond vector
                    seg = np.vstack((pos_i, pos_j))
                    cyl = trimesh.creation.cylinder(radius=bond_radius,
                                                    segment=seg, sections=24)
                    cyl.visual.face_colors = np.tile(bond_color, (num_bond_faces, 1))
                    meshes_cylinder.append(cyl)
    else: # k-d tree 
        tree = KDTree(coords, leafsize=10)
        for i in range(n):
            _nearest_coords, nearest_index = tree.query(coords[i], k=9) # 8 neighbors
            for j in nearest_index[1:]: # Skip itself
                d = mic_distance(coords[i], coords[j], box_length)
                thr = 1.2 * (cov_r[i] + cov_r[j])
                if d < thr:
                    # Unwrap j relative to i
                    disp = mic_vector(coords[j] - coords[i], box_length)
                    pos_i = np.mod(coords[i], box_length)
                    pos_j = pos_i + disp # may fall outside box but correct bond vector
                    seg = np.vstack((pos_i, pos_j))
                    cyl = trimesh.creation.cylinder(radius=bond_radius,
                                                    segment=seg, sections=24)
                    cyl.visual.face_colors = np.tile(bond_color, (num_bond_faces, 1))
                    meshes_cylinder.append(cyl)

    # --- Merge all into one mesh ---
    molecule = trimesh.util.concatenate(meshes_sphere.tolist() + meshes_cylinder)
    return molecule

## Molecules to meshes

Helper function to parallelize processing a single molecule and return in `{mol_id: mesh_object}` format. 

In [9]:
def process_single_molecule(mol_item, box_length, vdw, cov, sphere_radius_scale, sphere_subdiv, bond_radius):
    """Process a single molecule and return (mol_id, mesh)"""
    mol_id, atoms = mol_item
    elements = [infer_element(name) for name, _ in atoms]
    coords = np.vstack([pos for _, pos in atoms])
    # print(f'Molecule: {mol_id}')
    
    mesh = build_molecule_ballstick(
        coords, elements, vdw, cov, box_length,
        sphere_radius_scale=sphere_radius_scale,
        sphere_subdiv=sphere_subdiv,
        bond_radius=bond_radius
    )
    return mol_id, mesh

Function to parallelize processing all 1501 molecules.

In [10]:
from tqdm import tqdm


def molecules_to_meshes_parallel(molecules, box,
                                vdw, cov,
                                sphere_radius_scale=0.3,
                                sphere_subdiv=2,
                                bond_radius=0.1,
                                num_processes=None):
    """
    Convert parsed molecules into trimesh meshes.

    Parameters
    ----------
    molecules : dict[int, list[tuple]]
        From parse_gro(): molecules[i] = [(atomname, coords), ...]
        coords must be in Å
    box : np.ndarray
        Simulation box (Å), shape (3,) for orthorhombic or (3,3) for triclinic
    vdw, cov : dict
        Van der Waals and covalent radii
    sphere_radius_scale : float
        Scaling factor for atom radii
    sphere_subdiv : int
        Subdivisions for icosphere (mesh resolution)
    bond_radius : float
        Cylinder radius for bonds
    num_processes: int/None
        Number of CPU cores to use (all if not specified)

    Returns
    -------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary of molecule meshes keyed by mol_id
    """
    # assume orthorhombic box for now
    if box.shape == (3,):
        box_length = box
    else:
        raise NotImplementedError("Triclinic box handling not yet implemented")
    
    # --- Multiprocessing ---
    # number of processes (use all CPUs if not specified)
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    # partial function with fixed parameters
    process_func = partial(
        process_single_molecule,
        box_length=box_length,
        vdw=vdw,
        cov=cov,
        sphere_radius_scale=sphere_radius_scale,
        sphere_subdiv=sphere_subdiv,
        bond_radius=bond_radius
    )

    # parallel execution
    mol_items = list(molecules.items())
    num_mol = len(mol_items)
    
    with mp.Pool(processes=num_processes) as pool:
        tqdm_iterator = tqdm(
            pool.imap(process_func, mol_items),
            total=num_mol,
            desc=f'Processing {num_mol} molecules with {num_processes} logical cores',
            colour='#7BC8F6'
        )
        
        mol_meshes = list(tqdm_iterator) # Initialise progress bar
        
    return dict(mol_meshes) # Return as dictionary

In [11]:
# Now mol_meshes is {1: Trimesh(...), 2: Trimesh(...), ..., 1501: Trimesh(...)}
mol_meshes = molecules_to_meshes_parallel(molecules, box, vdw, cov, num_processes=None)
print(len(mol_meshes))        # 1501
print(mol_meshes[1])          # trimesh.Trimesh object

Processing 1501 molecules with 8 logical cores: 100%|██████████| 1501/1501 [01:04<00:00, 23.28it/s]

1501
<trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>


## Export molecule meshes (`.ply` file)

Saving output molecule meshes into `.PLY` file.

In [ ]:
def export_single_mesh(args, directory, file_type):
    """Export a single mesh to a PLY file."""
    mol_id, mesh = args
    ply_file = os.path.join(directory, f'molecule_{mol_id:04d}.ply')
    mesh.export(ply_file, file_type)
    
    return None


def export_meshes_to_ply(mol_meshes, directory, file_type='ply', num_processes=None):
    """Export all meshes to PLY files in the specified directory."""
    # creates directory if it doesn't exist
    if not os.path.exists(directory): 
        os.makedirs(directory)
        
    # number of processes (use all CPUs if not specified)
    if num_processes is None: 
        num_processes = mp.cpu_count()
    
    # partial function with fixed parameters
    process_func = partial(
        export_single_mesh,
        directory=directory,
        file_type='ply'
    )
    
    # parallel execution
    total = len(mol_meshes)
    args = mol_meshes.items()
    with mp.Pool(processes=num_processes) as pool:
        tqdm_iterator = tqdm(
            pool.imap_unordered(process_func, args),
            total=total,
            desc=f'Exporting {total} meshes with {num_processes} logical cores',
            colour='#7BC8F6'
        )
        list(tqdm_iterator) # Initialise progress bar
        
    return None

In [13]:
export_meshes_to_ply(mol_meshes, directory='./molecule_meshes', file_type='ply', num_processes=None)

Exporting 1501 meshes with 8 logical cores: 100%|██████████| 1501/1501 [00:06<00:00, 229.63it/s]


# Import mol_meshes

Import meshes from `.ply` files

In [7]:
def count_ply(path):
    """Count the total number of PLY files in a directory."""
    count = 0
    with os.scandir(path) as entries:
        for entry in entries:
            if entry.is_file() and entry.name.endswith('.ply'):
                count += 1
    return count

def find_ply(path):
    """Generator yielding the full path of all PLY files in a directory"""
    with os.scandir(path) as entries:
        for entry in entries:
            if entry.is_file() and entry.name.endswith('.ply'): 
                yield entry.path

def load_single_mesh(args):
    """Worker function to load a single mesh.
    
    This function is designed to be called by a multiprocessing pool.
    It takes a tuple of (mol_id, file_path) and returns a
    (mol_id, mesh) tuple.
    """
    mol_id, ply_file = args
    mesh = trimesh.load(ply_file, file_type='ply', process=True)
    return mol_id, mesh

def load_meshes_from_ply(directory, num_processes=None):
    """
    Load all PLY files from a directory into a dictionary of trimesh objects
    using multiprocessing.
    """    
    # Determine the number of processes
    if num_processes is None: 
        num_processes = mp.cpu_count()
    
    # Get the total number of files to show progress
    total_files = count_ply(directory)
    
    # Create the iterable of arguments for the workers
    args = enumerate(find_ply(directory))
    
    with mp.Pool(processes=num_processes) as pool:
        # Use imap_unordered for a lazy, memory-efficient map
        tqdm_iterator = tqdm(
            pool.imap_unordered(load_single_mesh, args),
            total=total_files,
            desc=f'Loading {total_files} meshes with {num_processes} cores',
            colour='#7BC8F6'
        )
        
        # Iterate over the results from the pool and populate the dictionary
        meshes = {mol_id: mesh for mol_id, mesh in tqdm_iterator}
            
    return meshes

In [8]:
mol_meshes = load_meshes_from_ply("./molecule_meshes")
mol_meshes[1]

Loading 1501 meshes with 8 cores: 100%|██████████| 1501/1501 [00:08<00:00, 181.25it/s]


<trimesh.Trimesh(vertices.shape=(18213, 3), faces.shape=(35904, 3))>

# Visualisation

In [9]:
# scene = trimesh.Scene(list(mol_meshes.values())[0:30])
# scene.show()

# Blocking Molecules

## Nearest Neighbour Candidate

In [9]:
def compute_centroids_and_radii_pbc(mol_meshes, box):
    """
    Compute periodic-boundary-condition (PBC) aware centroids and radii for a set of molecules.

    For each molecule mesh, this function:
      1. Selects a reference vertex (atom) as the origin.
      2. Unwraps all other vertices relative to this reference using the minimum-image convention (MIC),
         so that all atoms are locally unwrapped and contiguous in space.
      3. Computes the centroid (geometric center) of the unwrapped coordinates.
      4. Calculates the maximum MIC distance from the centroid to any vertex, defining the molecule's effective radius.

    Parameters
    ----------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their corresponding trimesh mesh objects.
    box : array-like, shape (3,) or (3,3)
        Simulation box dimensions (in Å). Should be a 3-element array for orthorhombic boxes.

    Returns
    -------
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å), unwrapped in PBC.
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å), defined as the maximum MIC distance
        from the centroid to any vertex in the molecule.
    """
    centroids, radii = {}, {}
    # box = np.array(box, dtype=float)

    for mol_id, mesh in mol_meshes.items():
        verts = mesh.vertices
        ref = verts[0]  # reference atom
        disp = mic_vector(verts - ref, box)
        unwrapped = ref + disp

        # centroid in unwrapped space
        center = unwrapped.mean(axis=0)

        # MIC distances from centroid to each vertex
        disp_center = mic_vector(verts - center, box)
        radius = np.linalg.norm(disp_center, axis=1).max()

        centroids[mol_id] = center
        radii[mol_id] = radius

    return centroids, radii


In [10]:
def wrap_points(points, box):
    """
    Wrap points into the primary simulation box using periodic boundary conditions.

    Each coordinate of the input points is wrapped into the interval [0, box_length)
    by applying the modulo operation with respect to the box dimensions. This ensures
    that all points are mapped inside the simulation box, consistent with periodic boundary conditions.

    Parameters
    ----------
    points : np.ndarray
        Array of points to wrap. Can be shape (N, 3) for N points in 3D, or any shape compatible with box.
    box : float or np.ndarray
        Simulation box dimensions. Can be a scalar (for cubic box) or array-like of shape (3,) for orthorhombic box.

    Returns
    -------
    wrapped_points : np.ndarray
        Array of wrapped points with the same shape as input, all coordinates in [0, box_length).
    """
    box = np.array(box, dtype=float)
    return points % box

def nearest_neighbors(mol_meshes, box, centroids, k=10, return_meshes=False):
    """
    Find the k nearest neighbors of each molecule, based on centroid distance.
    
    Parameters
    ----------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary of molecule meshes.
    box : float or array-like
        Simulation box length (for PBC). Pass a scalar if cubic.
    k : int
        Number of nearest neighbors to return per molecule.

    Returns
    -------
    dict[int, list[tuple[int,float]]]
        Mapping mol_id -> list of (neighbor_id, distance).
    """

    ids = list(centroids.keys())
    coords = np.vstack([centroids[i] for i in ids])
    coords_wrapped = wrap_points(coords, box)
    kd = KDTree(coords_wrapped, boxsize=box)

    neighbors = {}
    for idx, mol_id in enumerate(ids):
        dists, idxs = kd.query(coords[idx], k=k+1)
        dists, idxs = dists[1:], idxs[1:]
        if return_meshes:
            neighbors[mol_id] = [(mol_meshes[ids[j]], float(d)) for j, d in zip(idxs, dists)]
        else:
            neighbors[mol_id] = [(ids[j], float(d)) for j, d in zip(idxs, dists)]
    return neighbors


In [11]:
centroids, radii = compute_centroids_and_radii_pbc(mol_meshes, box)

neighbors = nearest_neighbors(mol_meshes, box, centroids, k=10, return_meshes=False)

neighbor_candidates = [(key, t[0]) for key, value in neighbors.items() for t in value]

neighbor_candidates_sorted = [(min(key, t[0]), max(key, t[0])) 
                      for key, value in neighbors.items() for t in value if key < t[0]]

neighbor_candidates_sorted = list(set(neighbor_candidates_sorted))
neighbor_candidates_sorted.sort()

In [12]:
print(centroids)
print(radii)
print(neighbors)
print(neighbor_candidates_sorted)

{3: TrackedArray([ 6.27323485, 11.06023064,  0.37734525]), 0: TrackedArray([12.93634966,  2.74190821,  2.57491574]), 2: TrackedArray([115.60540363, 108.61329486,  10.40924277]), 7: TrackedArray([30.43774116, 69.49291091, 69.94513842]), 4: TrackedArray([ 56.07153934, 110.82547847,  54.41995992]), 5: TrackedArray([90.67360083, 67.63296854, 10.90619973]), 6: TrackedArray([67.47546754, 99.40965556, 66.96492687]), 1: TrackedArray([116.29357705,   4.48886015,  -0.12691471]), 8: TrackedArray([77.687333  , 94.02462809, 80.49950726]), 9: TrackedArray([14.76492474, 90.75188038, 24.73613492]), 10: TrackedArray([74.90061524, 97.63711223, 24.77957535]), 11: TrackedArray([72.04929413, 64.77062183, 30.85434082]), 12: TrackedArray([87.40218468, 69.27247035, 68.11347499]), 13: TrackedArray([ 0.79612171, -1.47603447, 51.27537769]), 16: TrackedArray([ 7.94389914, 49.28790158, 87.15396399]), 15: TrackedArray([92.8843442 , 25.85622445, 83.67902497]), 17: TrackedArray([118.76096878, 113.71779894, 105.158830

## Blocking Algorithm 2

In [13]:
def blocked_by_any_2(ij_pair, shared_data):
    """
    Determine if the direct path between two molecule centroids is obstructed by any other molecule.

    For a given pair of molecules (i, j), this function checks whether the straight line
    connecting their centroids is intersected ("blocked") by any other molecule in the system.
    The check is performed in two steps:
      1. Fast sphere rejection: For each candidate blocking molecule, if its centroid is not
         within its effective radius of the line segment, it is skipped.
      2. Ray-mesh intersection: If the sphere check passes, a ray-mesh intersection test is
         performed to determine if the mesh of the candidate molecule blocks the path.

    Periodic boundary conditions (PBC) are handled using the minimum-image convention.

    Parameters
    ----------
    ij_pair : tuple[int, int]
        IDs of the two molecules to test for a direct connection.
    shared_data : dict
        Shared data dictionary containing the following:    

        centroids : dict[int, np.ndarray]
            Dictionary mapping molecule IDs to their centroid coordinates (in Å).
        radii : dict[int, float]
            Dictionary mapping molecule IDs to their effective radii (in Å).
        mol_meshes : dict[int, trimesh.Trimesh]
            Dictionary mapping molecule IDs to their trimesh mesh objects.
        ids : list[int]
            List of molecule IDs.

    Returns
    -------
    blocked : bool
        True if the path between i and j is blocked by any other molecule, False otherwise.
    """
    # Map molecule IDs to their index in ids
    i, j = ij_pair
    centroids, radii, mol_meshes = shared_data['centroids'], shared_data['radii'], shared_data['mol_meshes']
    ci, cj = centroids[i], centroids[j]
    seg_vec = cj - ci
    seg_len = np.linalg.norm(seg_vec)
    if seg_len < 1e-6: # ignores itself
        return False
    direction = seg_vec / seg_len

    # Get candidate molecule IDs (not indices)
    cand_ids = [t[1] for t in neighbor_candidates if t[0] == i]

    for mol_k in cand_ids:
        if mol_k in (i, j):
            continue

        # Quick sphere reject
        ck = centroids[mol_k]
        v = cj - ci
        w = ck - ci
        proj = np.dot(w, v) / np.dot(v, v)
        proj = np.clip(proj, 0.0, 1.0)
        closest = ci + proj * v
        if np.linalg.norm(ck - closest) > radii[mol_k]:
            continue

        # Expensive ray test
        if mol_meshes[mol_k].ray.intersects_any(
            ray_origins=ci.reshape(1, 3),
            ray_directions=direction.reshape(1, 3)
        ):
            return True
    return False

In [ ]:
def find_neighbors_2(centroids, radii, mol_meshes, box, neighbor_candidates, num_processes=None):
    neighbor_pairs = []
    """
    Determine all unblocked neighbor pairs from a list of candidate molecule pairs.

    Parameters
    ----------
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    box : np.ndarray
        Simulation box dimensions (in Å).
    neighbor_candidates : list[tuple[int, int]]
        List of candidate neighbor pairs (i, j) to test for blocking.

    Returns
    -------
    neighbor_pairs : list[tuple[int, int]]
        List of unblocked neighbor pairs (i, j).
    """
    
    ids = list(centroids.keys())
    
    # Determine the number of processes
    if num_processes is None: 
        num_processes = mp.cpu_count()
    
    # -- Multiprocessing ---
    # Manager keeps large shared data rather than creating multiple copies
    def run_parallel_manager():
        with mp.Manager() as manager:
            # Create a single Manager-backed dictionary to hold ALL constant data
            shared_data_proxy = manager.dict({
                'mol_meshes': mol_meshes,
                'centroids': centroids,
                'radii': radii,
                'ids': ids,
                'neighbor_candidates': neighbor_candidates
            })

            # The partial function only needs the shared proxy object
            worker_func = partial(blocked_by_any_2, shared_data=shared_data_proxy)
            with mp.Pool(processes=num_processes) as pool:
                results = pool.map(worker_func, neighbor_candidates)
                neighbor_pairs = [pair for pair, is_blocked in zip(neighbor_candidates, results) 
                                  if not is_blocked]  # Keep unblocked pairs
                return neighbor_pairs
                
    # Run the parallel manager and return results
    neighbor_pairs = run_parallel_manager()
    return neighbor_pairs

In [ ]:
results_2 = find_neighbors_2(centroids, radii, mol_meshes, box, neighbor_candidates_sorted)